# AI Employee Monitoring - Kaggle training

GPU + Internet required (Settings -> Accelerator -> GPU T4 x2 or P100).

Steps:
1. Upload your office videos as a Kaggle dataset and attach it (path shows under *Input*).
2. Set `REPO_URL` and `VIDEOS` below and run the cells in order.
3. Review `review.csv`, fix wrong labels, rebuild the folders.
4. Train the classifier and download `activity_classifier.onnx` + `.json`.
5. Put both files in `models/` of your repo and set `activity.classifier_model`.

Optional: fine-tune YOLO with `training/train_yolo.py` if COCO person detection is not enough.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
REPO_URL = 'https://github.com/YOUR_USER/ai-detector.git'  # <- your repo
VIDEOS = '/kaggle/input/office-videos'                     # <- attached dataset
OUT = '/kaggle/working'

import os, subprocess
work = '/kaggle/working/ai-detector'
if not os.path.isdir(work):
    subprocess.run(['git', 'clone', REPO_URL, work], check=True)
os.chdir(work)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', 'torchvision'], check=True)
print(os.getcwd())

In [ ]:
# ~15-30 crops per video is plenty; raise stride for long videos
!python training/prepare_activity_dataset.py --videos {VIDEOS} --out {OUT}/activity --stride 15

## Review labels

Open `/kaggle/working/activity/review.csv` in the Kaggle file browser (double click).
Set the `label` column to `working`, `phone` or `idle` (allowed values only).
Copy a few working crops into a new `idle` folder by relabelling them - a person
leaning back / away from the screen is a useful third class.

Then run the next cell to move the files into `crops/<label>/`.

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/working/activity/review.csv')
print(df['label'].value_counts())

# Example bulk fix: everything with conf < .5 and label 'phone' -> 'working'
# df.loc[(df['label'] == 'phone') & (df['conf'] < 0.5), 'label'] = 'working'
# df.to_csv('/kaggle/working/activity/review.csv', index=False)

!python training/prepare_activity_dataset.py --from-csv {OUT}/activity/review.csv --out {OUT}/activity

In [ ]:
!ls /kaggle/working/activity/crops/* | head -n 5
from PIL import Image
import glob, matplotlib.pyplot as plt

files = glob.glob('/kaggle/working/activity/crops/working/*.jpg')[:6]
files += glob.glob('/kaggle/working/activity/crops/phone/*.jpg')[:6]
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax in axes.ravel():
    ax.axis('off')
for ax, path in zip(axes.ravel(), files):
    ax.imshow(Image.open(path)); ax.set_title(path.split('/')[-2])
plt.tight_layout()

In [ ]:
!python training/train_activity_classifier.py \
    --data {OUT}/activity/crops \
    --arch mobilenet_v3_small --img-size 224 \
    --epochs 12 --batch 64 --lr 3e-4 \
    --out {OUT}/activity_model \
    --export-onnx {OUT}/activity_model/activity_classifier.onnx

In [ ]:
# sanity check the exported model
import json, numpy as np, onnxruntime as ort

sess = ort.InferenceSession('/kaggle/working/activity_model/activity_classifier.onnx')
labels = json.load(open('/kaggle/working/activity_model/activity_classifier.json'))['labels']
x = np.random.rand(1, 3, 224, 224).astype('float32')
logits = sess.run(None, {sess.get_inputs()[0].name: x})[0][0]
print(labels, logits)

!ls -lh /kaggle/working/activity_model/

## Download and integrate

1. Download `activity_classifier.onnx` and `activity_classifier.json` from the
   *Output* panel (or `/kaggle/working/activity_model/`).
2. Commit them into `models/` of your repo.
3. In `config.yaml`:

```yaml
activity:
  classifier_model: models/activity_classifier.onnx
  classifier_threshold: 0.6
```

4. Push and reboot the Streamlit app. Keep `yolov8n.pt` as the detector.

If person detection misses seated people, fine-tune YOLO as well:

```bash
python training/train_yolo.py --data /kaggle/input/office-yolo/data.yaml --epochs 50 --imgsz 960
```

then upload `best.pt` to `models/` and set `detector.person_model: models/best.pt`.